ATé o momento o código está capturando imagem da webcan, fazendo marcações do rosto e extraindo a região da boca

In [2]:
import cv2
import dlib
import numpy as np
import imutils
from imutils import face_utils
import matplotlib.pyplot as plt
import os
import argparse
import sys

In [3]:
def capturar_imagem():  #captura imagem da pessoa com a webcam
    cap = cv2.VideoCapture(0) #seleciona a webcan padrão
    print("Pressione [s] para salvar as imagens ou [q] para sair.")
    
    cont_salvar = 0
    if not cap.isOpened():
        print("Cannot open camera")
        exit()
    
    while True:
        # Capture frame-by-frame
        ret, frame = cap.read()
        # if frame is read correctly ret is True
        if not ret:
            print("Can't receive frame (stream end?). Exiting ...")
            break

        cv2.imshow('Imagem', frame) #prepara a janela
    
        key = cv2.waitKey(1) & 0xFF #desenha o frame na tela
    
        if key == ord('s'):
            cv2.imwrite("../static/sua_foto.jpg", frame)
    
        elif key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()  
    cv2.waitKey(1) 

In [5]:
capturar_imagem()

Pressione [s] para salvar as imagens ou [q] para sair.


In [4]:
# coding: utf-8
# import the necessary packages
from imutils import face_utils
import numpy as np
import dlib
import cv2
import matplotlib.pyplot as plt

class DetectFace:
    def __init__(self, image):
        # initialize dlib's face detector (HOG-based)
        # and then create the facial landmark predictor
        self.detector = dlib.get_frontal_face_detector()
        self.predictor = dlib.shape_predictor('../shape_predictor_68_face_landmarks.dat')

        #face detection part
        self.img = cv2.imread(image)
        #if self.img.shape[0]>500:
        #    self.img = cv2.resize(self.img, dsize=(0,0), fx=0.8, fy=0.8)

        # init face parts
        self.right_eyebrow = []
        self.left_eyebrow = []
        self.right_eye = []
        self.left_eye = []
        self.left_cheek = []
        self.right_cheek = []

        # detect the face parts and set the variables
        self.detect_face_part()


    # return type : np.array
    def detect_face_part(self):
        gray = cv2.cvtColor(self.img, cv2.COLOR_BGR2GRAY)
        
        # 1. Detecta todos os rostos encontrados
        rects = self.detector(gray, 1)
        
        # 2. Verificação de segurança: se a lista estiver vazia, avisa o usuário
        if len(rects) == 0:
            raise Exception("Não foi possível detectar um rosto na imagem. Tente uma foto mais clara.")
        
        # 3. Pega apenas o PRIMEIRO rosto da lista (índice 0)
        rect = rects[0]
        
        # Agora o predictor recebe o rosto específico (rect), não a lista (rects)
        shape = self.predictor(gray, rect)
        shape = face_utils.shape_to_np(shape)

        face_parts = []
        for (name, (i, j)) in face_utils.FACIAL_LANDMARKS_IDXS.items():
            face_parts.append(shape[i:j])
            
        face_parts = face_parts[1:5]
        
        self.right_eyebrow = self.extract_face_part(face_parts[0])
        self.left_eyebrow = self.extract_face_part(face_parts[1])
        self.right_eye = self.extract_face_part(face_parts[2])
        self.left_eye = self.extract_face_part(face_parts[3])
        
        # Bochechas baseadas nos pontos do nariz (29, 33) e contorno (4, 48 / 54, 12)
        self.left_cheek = self.img[shape[29][1]:shape[33][1], shape[4][0]:shape[48][0]]
        self.right_cheek = self.img[shape[29][1]:shape[33][1], shape[54][0]:shape[12][0]]


    # parameter example : self.right_eye
    # return type : image
    def extract_face_part(self, face_part_points):
        (x, y, w, h) = cv2.boundingRect(face_part_points)
        crop = self.img[y:y+h, x:x+w]
        adj_points = np.array([np.array([p[0]-x, p[1]-y]) for p in face_part_points])

        # Create an mask
        mask = np.zeros((crop.shape[0], crop.shape[1]))
        cv2.fillConvexPoly(mask, adj_points, 1)
        mask = mask.astype(bool)
        crop[np.logical_not(mask)] = [255, 0, 0]

        return crop

In [26]:
# def marcacoes_faciais():
#     sys.argv = ['']  #fazendo marcações faciais com o landmarks.dat
    
#     ap = argparse.ArgumentParser()
#     ap.add_argument("-p", "--shape-predictor", required=False,
#         help="path to facial landmark predictor", 
#         default="../shape_predictor_68_face_landmarks.dat")
    
    
#     ap.add_argument("-i", "--image", required=False, 
#         help="path to input image", 
#         default="../static/sua_foto.jpg") 
    
#     args = vars(ap.parse_args())
    
#     # Inicializa o detector e o preditor
#     detector = dlib.get_frontal_face_detector()
#     predictor = dlib.shape_predictor(args["shape_predictor"])
    
#     # Carrega a imagem
#     image = cv2.imread(args["image"])
    
#     if image is None:
#         print(f"Erro: Não foi possível encontrar a imagem em: {args['image']}")
#     else:
#         image = imutils.resize(image, width=500)
#         gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#         rects = detector(gray, 1)
    
#         for (i, rect) in enumerate(rects):
#             shape = predictor(gray, rect)
#             shape = face_utils.shape_to_np(shape) #convertendo cada parte em um np array
#             # Aqui você pode desenhar os pontos ou processar o shape
#             print(f"Rosto {i+1} detectado!")

#     for (name, (i,j)) in face_utils.FACIAL_LANDMARKS_IDXS.items(): 
#         if (name == "mouth"):
#             # extract the ROI of the face region as a separate image
#             (x1, y1, w1, h1) = cv2.boundingRect(np.array([shape[i:j]]))
#             roi = image[y1:y1 + h1, x1:x1 + w1]
#             roi = imutils.resize(roi, width=250, inter=cv2.INTER_CUBIC)
#             # show the particular face part
#             cv2.imshow("mouth", roi)
#             cv2.imwrite("../result/face_mouth_pic.jpg", roi)


In [20]:
print(face_utils.FACIAL_LANDMARKS_IDXS.items())

odict_items([('mouth', (48, 68)), ('inner_mouth', (60, 68)), ('right_eyebrow', (17, 22)), ('left_eyebrow', (22, 27)), ('right_eye', (36, 42)), ('left_eye', (42, 48)), ('nose', (27, 36)), ('jaw', (0, 17))])
